# Day 4 · AI가 제안한 주문도 같은 문지기를 지납니다

**여는 법:** `런타임 → 모두 실행`. 새 세션이면 첫 Cell부터.
**기대 결과:** 제안 5건 중 통과 3건, 거부 2건 (`녹차라떼 한 잔`, `라떼 두 잔`). 기대표 검사 Cell은 아무 출력 없이 지나갑니다.
**한계:** `suggest_order`는 표에 있는 다섯 문장만 압니다. 다른 문장은 `KeyError`로 멈춥니다.

In [ ]:
# ── 첫 Cell · 준비 ──────────────────────────────────────────
# Colab은 구글이 빌려주는 컴퓨터입니다. 새로 켤 때마다 빈 컴퓨터이므로
# 필요한 파일을 GitHub에서 받아 와야 합니다. 이 Cell이 그 일을 합니다.
#
# 실행하는 법: 이 Cell을 클릭한 뒤 왼쪽 ▶ 버튼을 누르거나 Shift+Enter.
# 위 메뉴 [런타임 → 모두 실행]을 누르면 위에서 아래로 전부 실행됩니다.

import os, sys          # import = 파이썬에 이미 들어 있는 도구 상자를 꺼내는 일

# 줄 앞의 ! 는 "파이썬이 아니라 터미널 명령"이라는 표시입니다.
# git clone = GitHub에 있는 폴더를 통째로 이 컴퓨터로 복사하는 명령.
if not os.path.isdir("jnu-llmops-precourse-day2"):      # 이미 받았으면 건너뜁니다
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 파이썬이 import 할 파일을 찾는 폴더 목록에 어제의 solution 폴더를 넣습니다.
# 이 줄이 없으면 아래 Cell의 from order import Order 가 파일을 못 찾습니다.
sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")

## 1. 어제의 문지기를 그대로 가져옵니다

In [ ]:
import json
from catalog import MENU
from order import Order
from pricing import calculate_bill

REQUIRED_KEYS = ("order_id", "items", "is_student")

def check_order(record):
    for key in REQUIRED_KEYS:                       # 1. 필수 Key
        if key not in record:
            return False, f"필수 Key 없음: {key}"
    for item in record["items"]:
        if item["menu_name"] not in MENU:           # 2. 허용 메뉴
            return False, f"없는 메뉴: {item['menu_name']}"
        quantity = item["quantity"]
        if type(quantity) is not int or not 1 <= quantity <= 10:   # 3. 수량 범위
            return False, f"수량 범위 밖: {quantity!r}"
    return True, ""


## 2. `suggest_order`는 응답이 정해진 대역입니다
실제 모델 대신, 문장마다 답이 정해진 표를 씁니다. 같은 문장이면 언제나 같은 제안이 나옵니다.

In [ ]:
SUGGESTIONS = {
    "라떼 두 잔, 학생이에요": {"order_id": "S01", "items": [{"menu_name": "카페라떼", "quantity": 2}], "is_student": True},
    "아메리카노 하나요":       {"order_id": "S02", "items": [{"menu_name": "아메리카노", "quantity": 1}], "is_student": False},
    "초코라떼 셋, 학생 할인":  {"order_id": "S03", "items": [{"menu_name": "초코라떼", "quantity": 3}], "is_student": True},
    "녹차라떼 한 잔":          {"order_id": "S04", "items": [{"menu_name": "녹차라떼", "quantity": 1}], "is_student": False},
    "라떼 두 잔":              {"order_id": "S05", "items": [{"menu_name": "카페라떼", "quantity": "두"}], "is_student": False},
}

def suggest_order(text):
    return SUGGESTIONS[text]

for text in SUGGESTIONS:
    ok, reason = check_order(suggest_order(text))
    print(f"{text!r:22} -> {ok} {reason}")

## 3. 기대표로 한 번에 검사합니다
기대는 실행 전에 적습니다. 다섯 건이 모두 맞으면 이 Cell은 아무것도 출력하지 않습니다.

In [ ]:
with open("jnu-llmops-precourse-day3/data/expected_day4.json", encoding="utf-8") as f:
    expected = json.load(f)

for text, want in expected.items():
    ok, reason = check_order(suggest_order(text))
    assert ok == want, (text, ok, reason)

## 4. 거부는 재시도가 아니라 기록과 사람에게 넘기기입니다

In [ ]:
rejected = []
for text in SUGGESTIONS:
    ok, reason = check_order(suggest_order(text))
    if not ok:
        rejected.append({"text": text, "reason": reason})
print(json.dumps(rejected, ensure_ascii=False, indent=2))

## 5. 표에 없는 문장은 멈춥니다
아래 Cell은 오류를 붙잡아 출력만 합니다.

In [ ]:
try:
    suggest_order("녹차 프라페")
except KeyError as e:
    print("KeyError:", e)

## 확장 (빠른 학생)
`expected` 에 없는 문장을 하나 골라 `SUGGESTIONS` 와 `expected_day4` 에 **둘 다** 추가하고, 기대표 검사 Cell을 다시 실행합니다. 어느 쪽을 먼저 적어야 할까요?

## README — 다른 사람이 이 노트북을 열 때
- 여는 법:
- 기대 결과:
- 한계: